In [38]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader,TensorDataset
import torch.nn as nn
import optuna
import torch.optim as optim

### Preparing Dataset

In [15]:
X,y = make_classification(n_samples=1000,n_features=20,n_classes=2,random_state=42)

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [25]:
X_train[0],y_train[0].item()

(array([ 0.50363664, -1.51368248, -0.46907062,  1.90176571, -0.87064279,
         1.82004715,  1.66291365,  1.29105223, -0.16713608, -1.04718436,
         1.43003039,  0.20104766,  1.27577182, -1.13260729,  1.75008532,
        -1.4089039 ,  0.03301588, -0.80340946, -1.31410638,  1.41209637]),
 1)

In [26]:
X_train_tensor,X_test_tensor = torch.tensor(X_train,dtype=torch.float32),torch.tensor(X_test,dtype=torch.float32)
y_train_tensor,y_test_tensor = torch.tensor(y_train,dtype=torch.long),torch.tensor(y_test,dtype=torch.long)

In [28]:
y_train_tensor[0]

tensor(1)

In [32]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [34]:
class SimpleNN(nn.Module):
    def __init__(self,input_neu,hidden_neu):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_neu,hidden_neu),
            nn.ReLU(),
            nn.Linear(hidden_neu,2)
        )
    def forward(self,x):
        return self.network(x)

In [53]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
image,label = next(iter(train_loader))
len(image),len(label),len(train_loader),image.size(0)


(32, 32, 25, 32)

In [58]:
def objective(trial):
    hidden_new = trial.suggest_int('hidden_new',16,128)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    model = SimpleNN(input_neu=20,hidden_neu=hidden_new)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),lr = learning_rate)

    batch_size = 32
    train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    test_loader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

    epochs = 20
    for epoch in range(epochs):
        model.train()
        for X_batch,y_batch in train_loader:
            preds = model(X_batch)
            loss = criterion(preds,y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    model.eval()
    all_pred = 0
    correct_pred = 0
    
    for X_batch,y_batch in test_loader:
        outputs = model(X_batch)
        _,pred = torch.max(outputs,1)
        all_pred += len(y_batch)
        correct_pred += (pred == y_batch).sum().item() 
    accuracy = correct_pred*100/all_pred
    return accuracy

In [59]:
study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=20)

[I 2026-05-26 23:56:42,599] A new study created in memory with name: no-name-90f29f0a-9d13-4984-9b87-38b6bc6a67bf
[I 2026-05-26 23:56:42,769] Trial 0 finished with value: 85.5 and parameters: {'hidden_new': 114, 'learning_rate': 0.00014377192969043423}. Best is trial 0 with value: 85.5.
[I 2026-05-26 23:56:42,873] Trial 1 finished with value: 80.0 and parameters: {'hidden_new': 29, 'learning_rate': 0.007160025767097962}. Best is trial 0 with value: 85.5.
[I 2026-05-26 23:56:42,980] Trial 2 finished with value: 84.5 and parameters: {'hidden_new': 75, 'learning_rate': 0.0002207960564453676}. Best is trial 0 with value: 85.5.
[I 2026-05-26 23:56:43,084] Trial 3 finished with value: 83.5 and parameters: {'hidden_new': 54, 'learning_rate': 0.004687362811464414}. Best is trial 0 with value: 85.5.
[I 2026-05-26 23:56:43,188] Trial 4 finished with value: 83.0 and parameters: {'hidden_new': 40, 'learning_rate': 0.007165325940225229}. Best is trial 0 with value: 85.5.
[I 2026-05-26 23:56:43,293]

In [ ]:
print(f"Best Parameter: {study.best_params()")